# Practical PyTorch · Part 17 — Companion Notebook

This goes with **"The Hub and the Heavyweights"**. Run the cells top to bottom (Shift+Enter).

**Turn the GPU on first:** *Runtime → Change runtime type → Hardware accelerator → GPU* (a free **T4** is fine). The quantization cell near the end *requires* a GPU and will error on a CPU-only runtime.

## 0. Install the extras

`transformers` for the models, `accelerate` for `device_map="auto"`, `bitsandbytes` for 4-bit quantization. The first run takes a minute.

In [ ]:
!pip install -q transformers accelerate bitsandbytes

## 1. What hardware did Colab give us?

Step one of fitting a model into memory is knowing how much memory you have. If this prints `GPU available: False`, go turn the GPU on (see the note at the top) and run it again.

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("VRAM (GB):", round(vram, 1))

## 2. The memory rule of thumb

By default each weight is a 32-bit float — **4 bytes**. So a model's rough footprint is `parameters × bytes-per-weight`. Watch how the same model gets cheaper as we spend fewer bits per weight.

In [ ]:
params = 7e9   # a 7-billion-parameter model

for label, bytes_per_weight in [("fp32 (default)", 4), ("fp16/bf16", 2), ("4-bit", 0.5)]:
    gb = params * bytes_per_weight / 1e9
    print(f"{label:>14}: ~{gb:5.1f} GB")

print("\nA free T4 has ~16 GB. Notice which rows fit.")

## 3. Finding a model on the Hub

Browse [huggingface.co/models](https://huggingface.co/models): **filter by task** (Text Generation, Image Classification, ...), **sort by downloads** for the battle-tested ones, then read the **model card** — what it does, its inputs/outputs, and its **license** (Apache-2.0 / MIT = commercial-friendly; non-commercial = demos only).

You can also list models from code. Here are popular, openly-licensed text-generation models — the kind of names you'd paste into the cells below:

In [ ]:
from huggingface_hub import list_models

models = list_models(task="text-generation", sort="downloads", direction=-1, limit=8)
for m in models:
    print(f"{m.id:<45} downloads={m.downloads:,}")

## 4. Gated models (optional)

Some models — Llama, Gemma — are **gated**: you accept the terms on the model's page, then log in with an access token so the Hub knows it's you.

Create a read token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). The tidy way in Colab: click the **key icon** in the left sidebar and add a secret named `HF_TOKEN` — `transformers` picks it up automatically. Or log in interactively:

In [ ]:
# Only needed for GATED models. Skip this if your model is openly available.
# from huggingface_hub import login
# login()   # paste your token when prompted
print("Skipping login — the example model below is open (no token needed).")

## 5. Lever 1+2+3: half precision + device_map="auto"

`dtype=torch.bfloat16` halves the memory (2 bytes/weight instead of 4) with no real quality cost for inference. `device_map="auto"` lets `accelerate` place the model on the GPU for you.

The model below — `Qwen/Qwen2.5-0.5B-Instruct` — is tiny (0.5B params) and Apache-2.0 licensed, so it runs anywhere. It's **illustrative**: the exact same two arguments are what make a 7B model fit on a T4.

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
)

messages = [{"role": "user", "content": "In one sentence, what is a GPU good for?"}]
out = pipe(messages, max_new_tokens=60)
print(out[0]["generated_text"][-1]["content"])

## 6. Lever 4: 4-bit quantization

Quantization stores each weight with **far fewer bits** (~0.5 bytes), shrinking VRAM use dramatically for a small quality cost. This is how people run large models on modest GPUs.

**Requirements:** `bitsandbytes` (installed above) **and a CUDA GPU** — there's no CPU fallback. Quantizing this 0.5B model is overkill, but the code is *identical* for a 7B+ model.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

assert torch.cuda.is_available(), "4-bit quantization needs a GPU. Runtime -> Change runtime type -> GPU."

bnb = BitsAndBytesConfig(load_in_4bit=True)

name = "Qwen/Qwen2.5-0.5B-Instruct"   # illustrative; reach for this with a 7B+ model
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name,
    quantization_config=bnb,
    device_map="auto",
)

footprint = model.get_memory_footprint() / 1e6
print(f"Loaded in 4-bit. Memory footprint: ~{footprint:.0f} MB")

## 7. Generate with the quantized model

Once loaded, a quantized model is used exactly like any other — tokenize the input, generate, decode.

In [ ]:
messages = [{"role": "user", "content": "Explain quantization to a curious 10-year-old, in two sentences."}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

generated = model.generate(inputs, max_new_tokens=80)
reply = tokenizer.decode(generated[0][inputs.shape[-1]:], skip_special_tokens=True)
print(reply)

## When the free GPU isn't enough

These levers have limits. A 4-bit 13B model fits on a T4; a 70B model doesn't (even in 4-bit it wants ~40 GB). When you hit the wall: reach for a **smaller model** (there's almost always a 7B that's 90% as good), a **bigger GPU** (Colab Pro's L4/A100), or a **hosted API** for the truly huge ones.

The skill is recognizing when you can't squeeze it in — and picking the right-sized model instead of the biggest one. Next up (Part 18): a debugging playbook — the errors you're most likely to hit, and the fixes.